# Phase 5 — Cross-Model Comparison (Direction 3)
**Project:** Temporal GNN + XAI for Anti-Money Laundering
**Purpose:** Aggregate the four Direction-3 models — Static GCN, Static GAT, Static
GraphSAGE, EvolveGCN-H (fixed) — into one benchmark: does the T43 SHAP-ranking drift
and the local→aggregated feature shift replicate across every architecture?

**Prerequisite:** notebooks 02, 03, and 04 (all four `MODEL_NAME` passes) must have
already run on Colab, so `data/processed/*_summary.json`, `data/shap/kendall_tau_*.json`,
`data/shap/local_vs_aggregated_*.json`, and `data/shap/shap_*_W4.pkl` all exist for
`gcn`, `gat`, `sage`, `evolvegcn_fixed`. This notebook does no training and no SHAP —
it only reads cached artifacts, so it is cheap to re-run.

### What this notebook produces
1. Benchmark table — test F1, pre/post-T43 F1, F1 drop, W3→W4 τ + CI + permutation p, per model
2. Grouped τ bar chart — 3 transitions × 4 models, bootstrap CI error bars
3. Local-fraction line chart — top-10 local-feature fraction per window, one line per model
4. Top-10 W4 feature agreement — pairwise Jaccard + whether `feat_143` ranks #1 everywhere
5. **Go/no-go gate** — printed explicitly as PASS/FAIL (see `PROGRESS.md` → "Phase 7")

---
## Cell 1 — Imports & Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import pickle
import itertools

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 11})

BASE_DIR      = '/content/drive/MyDrive/Capstone/Testing'
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
FIGURES_DIR   = os.path.join(BASE_DIR, 'figures')
SHAP_DIR      = os.path.join(BASE_DIR, 'data', 'shap')
os.makedirs(FIGURES_DIR, exist_ok=True)

MODELS       = ['gcn', 'gat', 'sage', 'evolvegcn_fixed']
MODEL_LABEL  = {'gcn': 'Static GCN', 'gat': 'Static GAT', 'sage': 'Static GraphSAGE',
                'evolvegcn_fixed': 'EvolveGCN-H (fixed)'}
SUMMARY_FILE = {'gcn': 'static_gcn_summary.json', 'gat': 'gat_summary.json',
                'sage': 'sage_summary.json', 'evolvegcn_fixed': 'evolvegcn_fixed_summary.json'}

TAU_THRESHOLD = 0.70   # HARD INVARIANT — must match notebook 04
GATE_TRANSITION = 'W3→W4'

print('Cross-model comparison for:', ', '.join(MODEL_LABEL[m] for m in MODELS))

---
## Cell 2 — Load Cached Artifacts
Fails loudly (missing-file error) if any of the four models hasn't finished notebooks
02/03/04 yet — that is the point: a partial benchmark should not silently pass the gate.

In [ ]:
summaries, taus, local_shifts, shap_w4 = {}, {}, {}, {}

for m in MODELS:
    with open(os.path.join(PROCESSED_DIR, SUMMARY_FILE[m])) as f:
        summaries[m] = json.load(f)
    with open(os.path.join(SHAP_DIR, f'kendall_tau_{m}.json')) as f:
        taus[m] = json.load(f)
    with open(os.path.join(SHAP_DIR, f'local_vs_aggregated_{m}.json')) as f:
        local_shifts[m] = json.load(f)
    with open(os.path.join(SHAP_DIR, f'shap_{m}_W4.pkl'), 'rb') as f:
        shap_w4[m] = pickle.load(f)
    print(f'{MODEL_LABEL[m]:<22} loaded: summary, τ (3 transitions), local/agg (4 windows), SHAP W4')

print('\nAll four models loaded.')

---
## Cell 3 — Benchmark Table

In [ ]:
def _tau_at(m, comparison):
    for r in taus[m]['results']:
        if r['comparison'] == comparison:
            return r
    return None

bench_rows = []
print(f'{"Model":<22}{"Test F1":>9}{"Pre-T43":>9}{"Post-T43":>10}{"Drop":>8}'
      f'{"W3\u2192W4 \u03c4":>10}{"95% CI":>18}{"perm p":>9}')
print('-' * 95)
for m in MODELS:
    s = summaries[m]
    t = _tau_at(m, GATE_TRANSITION)
    row = {
        'model': MODEL_LABEL[m], 'kind': m,
        'test_F1': s['test_metrics']['F1'],
        'pre_t43_f1': s['pre_t43_mean_f1'], 'post_t43_f1': s['post_t43_mean_f1'],
        'f1_drop': s['f1_drop'],
        'w3_w4_tau': t['tau'], 'w3_w4_ci_low': t['ci_low'], 'w3_w4_ci_high': t['ci_high'],
        'w3_w4_perm_p': t['perm_pvalue'], 'w3_w4_drift_flagged': t['drift'],
    }
    bench_rows.append(row)
    ci_str = f"[{row['w3_w4_ci_low']:.3f}, {row['w3_w4_ci_high']:.3f}]"
    print(f"{MODEL_LABEL[m]:<22}{row['test_F1']:>9.4f}{row['pre_t43_f1']:>9.4f}"
          f"{row['post_t43_f1']:>10.4f}{row['f1_drop']:>8.4f}"
          f"{row['w3_w4_tau']:>10.4f}{ci_str:>18}{row['w3_w4_perm_p']:>9.4f}")

bench_path = os.path.join(PROCESSED_DIR, 'direction3_benchmark_table.json')
with open(bench_path, 'w') as f:
    json.dump(bench_rows, f, indent=2)
print(f'\nSaved → {bench_path}')

---
## Cell 4 — Grouped τ Bar Chart
Headline: every architecture's SHAP ranking destabilises at the same transition.

In [ ]:
transitions = [r['comparison'] for r in taus[MODELS[0]]['results']]   # W1→W2, W2→W3, W3→W4
colors = {'gcn': '#4A90D9', 'gat': '#0D9488', 'sage': '#F59E0B', 'evolvegcn_fixed': '#E05252'}

fig, ax = plt.subplots(figsize=(11, 5.5))
n_models = len(MODELS)
width = 0.8 / n_models
x = np.arange(len(transitions))

for i, m in enumerate(MODELS):
    rows = [_tau_at(m, c) for c in transitions]
    vals = [r['tau'] for r in rows]
    lo   = [r['tau'] - r['ci_low']  for r in rows]
    hi   = [r['ci_high'] - r['tau'] for r in rows]
    pos  = x + (i - (n_models - 1) / 2) * width
    ax.bar(pos, vals, width=width * 0.9, color=colors[m], label=MODEL_LABEL[m])
    ax.errorbar(pos, vals, yerr=[lo, hi], fmt='none', ecolor='black', capsize=3, linewidth=1.1)

ax.axhline(y=TAU_THRESHOLD, color='black', linestyle='--', linewidth=1.6,
           label=f'Drift threshold (τ = {TAU_THRESHOLD})')
ax.set_xticks(x)
ax.set_xticklabels(transitions)
ax.set_ylabel('Kendall τ')
ax.set_title('SHAP Feature-Ranking Stability Across Architectures\n'
             'W3→W4 drop corresponds to the T43 dark-market shutdown',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'direction3_grouped_tau.png')
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

---
## Cell 5 — Local-Fraction Line Chart
Headline: the local→aggregated feature shift is a property of the data, not one model.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
window_order = ['W1', 'W2', 'W3', 'W4']

for m in MODELS:
    by_window = {w['window']: w['local_frac'] for w in local_shifts[m]['windows']}
    vals = [by_window[w] for w in window_order]
    ax.plot(window_order, vals, marker='o', linewidth=2.2, markersize=8,
            color=colors[m], label=MODEL_LABEL[m])

ax.axvline(x=2.5, color='red', linestyle='--', linewidth=1.5, alpha=0.6)
ax.text(2.55, 0.05, '← T43', color='red', fontsize=9)
ax.set_ylim(-0.02, 1.02)
ax.set_ylabel('Fraction of top-10 SHAP features that are local (feat_0–93)')
ax.set_title('Local→Aggregated Feature Reliance Across Architectures',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, 'direction3_local_fraction.png')
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

---
## Cell 6 — Top-10 W4 Feature Agreement (Jaccard)
If the post-shutdown feature set is a property of the data rather than of one
architecture, the four models' top-10 W4 features should overlap substantially, and
`feat_143` (the headline post-shutdown feature from the original two-model run) should
rank highly for all four.

In [ ]:
top10_w4 = {m: set(shap_w4[m]['top10_idx']) for m in MODELS}

print('TOP-10 W4 FEATURES PER MODEL')
for m in MODELS:
    print(f"  {MODEL_LABEL[m]:<22}: {shap_w4[m]['top10_names']}")

print('\nPAIRWISE JACCARD SIMILARITY (top-10 W4)')
print(f'{"":<22}' + ''.join(f'{MODEL_LABEL[m]:>22}' for m in MODELS))
jaccard = {}
for m1 in MODELS:
    row_str = f'{MODEL_LABEL[m1]:<22}'
    for m2 in MODELS:
        inter = len(top10_w4[m1] & top10_w4[m2])
        union = len(top10_w4[m1] | top10_w4[m2])
        j = inter / union if union else 0.0
        jaccard[f'{m1}__{m2}'] = round(j, 3)
        row_str += f'{j:>22.3f}'
    print(row_str)

feat_143_rank = {m: (shap_w4[m]['top10_idx'].index(143) + 1 if 143 in shap_w4[m]['top10_idx'] else None)
                 for m in MODELS}
print('\nfeat_143 rank in each model\'s top-10 W4 (1 = most important, None = not in top-10):')
for m in MODELS:
    print(f'  {MODEL_LABEL[m]:<22}: {feat_143_rank[m]}')

agreement_path = os.path.join(SHAP_DIR, 'direction3_w4_agreement.json')
with open(agreement_path, 'w') as f:
    json.dump({'top10_w4': {m: shap_w4[m]['top10_names'] for m in MODELS},
               'jaccard': jaccard, 'feat_143_rank': feat_143_rank}, f, indent=2)
print(f'\nSaved → {agreement_path}')

---
## Cell 7 — Go/No-Go Gate
Direction 3 **holds** if both conditions are true. See `PAPER_IDEAS.md` / `PROGRESS.md`
→ "Phase 7" for what happens on FAIL (fall back to Direction 1).

In [ ]:
# Condition A: every model flags drift (τ < threshold) at W3→W4
cond_a_per_model = {m: _tau_at(m, GATE_TRANSITION)['tau'] < TAU_THRESHOLD for m in MODELS}
cond_a = all(cond_a_per_model.values())

# Condition B: ≥3/4 models show top-10 local fraction dropping W3→W4
def _local_frac_at(m, window):
    return next(w['local_frac'] for w in local_shifts[m]['windows'] if w['window'] == window)

cond_b_per_model = {m: _local_frac_at(m, 'W4') < _local_frac_at(m, 'W3') for m in MODELS}
cond_b = sum(cond_b_per_model.values()) >= 3

gate_pass = cond_a and cond_b

print('GO/NO-GO GATE — DIRECTION 3')
print('=' * 60)
print(f'Condition A — all 4 models flag τ < {TAU_THRESHOLD} at {GATE_TRANSITION}:')
for m in MODELS:
    tau_val = _tau_at(m, GATE_TRANSITION)['tau']
    print(f'  {MODEL_LABEL[m]:<22}: τ={tau_val:.4f}  {"✓ drift" if cond_a_per_model[m] else "✗ stable"}')
print(f'  → {"PASS" if cond_a else "FAIL"} ({sum(cond_a_per_model.values())}/4 models flag drift)')
print()
print('Condition B — ≥3/4 models show local-fraction drop at W4 vs W3:')
for m in MODELS:
    w3, w4 = _local_frac_at(m, 'W3'), _local_frac_at(m, 'W4')
    print(f'  {MODEL_LABEL[m]:<22}: W3={w3:.2f} → W4={w4:.2f}  '
          f'{"✓ dropped" if cond_b_per_model[m] else "✗ did not drop"}')
print(f'  → {"PASS" if cond_b else "FAIL"} ({sum(cond_b_per_model.values())}/4 models show the drop)')
print()
print('=' * 60)
print(f'DIRECTION 3 GATE: {"PASS" if gate_pass else "FAIL"}')
print('=' * 60)
if gate_pass:
    print('Reasoning drift at T43 is universal across architectures.')
    print('Proceed with the Direction 3 write-up (PAPER_IDEAS.md).')
else:
    print('The "universal reasoning drift" headline does not hold as stated.')
    print('Fall back to Direction 1 (PAPER_IDEAS.md) — everything built here becomes')
    print('that paper\'s drift section, without the "universal" framing.')

gate_result = {
    'condition_a_all_drift': cond_a, 'condition_a_per_model': cond_a_per_model,
    'condition_b_majority_shift': cond_b, 'condition_b_per_model': cond_b_per_model,
    'gate_pass': gate_pass,
}
gate_path = os.path.join(PROCESSED_DIR, 'direction3_gate_result.json')
with open(gate_path, 'w') as f:
    json.dump(gate_result, f, indent=2)
print(f'\nSaved → {gate_path}')